# DevGen — ControlNet Training for Devanagari Handwriting (v5)

Trains a ControlNet on top of Stable Diffusion v1.5 to generate realistic Devanagari handwriting
conditioned on font-rendered text layout images.

### Key Updates in v5:
- **Fixed Validation Sampling DType Mismatch** — Uses in-memory components to prevent CPU/GPU mixed dtype crash
- **Dynamic Cross-Attention Prompts** — Uses phonetic English transliterations to guide character structures
- **Higher Learning Rate** — `1.5e-5` for faster zero-conv mapping convergence
- **Square 256x256 Architecture** — Perfectly aligned with SD latents

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════════════════
!pip install -q diffusers transformers accelerate datasets safetensors

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 2: Imports
# ══════════════════════════════════════════════════════════════════
import os
import gc
import time
import shutil
import traceback
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFont
from datasets import load_dataset
from accelerate import Accelerator
from accelerate.utils import set_seed, ProjectConfiguration
from transformers import AutoTokenizer, CLIPTextModel
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    UNet2DConditionModel,
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)
from diffusers.optimization import get_scheduler
import numpy as np

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3: Configuration
# ══════════════════════════════════════════════════════════════════
MODEL_ID        = "runwayml/stable-diffusion-v1-5"
IMG_SIZE        = 256              # Square images
BATCH_SIZE      = 6                # Safe on T4
GRADIENT_ACCUM  = 4                # Effective batch = 24
LEARNING_RATE   = 1.5e-5           # Speeds up ControlNet zero-conv opening
MAX_TRAIN_STEPS = 20000            # Safe limit
MAX_TIME        = 39000            # ~10.8 hours Kaggle safety timeout
SAVE_STEPS      = 500
LOG_STEPS       = 50
SAMPLE_STEPS    = 500              # Generate validation samples
WARMUP_STEPS    = 300
SEED            = 42
OUTPUT_DIR      = "controlnet_devanagari_v4"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/samples", exist_ok=True)

# ─── Font Discovery ───
FONT_FILE = None
FONT_SEARCH_PATHS = [
    "font.ttf",
    "NotoSansDevanagari-Regular.ttf",
    "backend/font.ttf",
    "../font.ttf",
    "/kaggle/working/font.ttf",
    "/kaggle/working/NotoSansDevanagari-Regular.ttf",
]
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith((".ttf", ".ttc")):
                FONT_SEARCH_PATHS.append(os.path.join(root, f))

for path in FONT_SEARCH_PATHS:
    if os.path.exists(path):
        FONT_FILE = path
        break

if FONT_FILE is None:
    import urllib.request
    url = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf"
    FONT_FILE = "NotoSansDevanagari-Regular.ttf"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as r, open(FONT_FILE, "wb") as f:
            f.write(r.read())
    except Exception as e:
        print(f"❌ Font download failed: {e}")
        FONT_FILE = None

print(f"Using Font: {FONT_FILE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4: Dataset & Transliteration Helper
# ══════════════════════════════════════════════════════════════════

def transliterate_devanagari(text):
    """Transliterates Devanagari Hindi/Nepali words phonetically into Latin characters."""
    char_map = {
        'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'उ': 'u', 'ऊ': 'oo', 'ऋ': 'ri', 'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
        'ा': 'aa', 'ि': 'i', 'ी': 'ee', 'ु': 'u', 'ू': 'oo', 'ृ': 'ri', 'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au',
        'ं': 'm', 'ः': 'h', 'ँ': 'n',
        'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
        'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'yn',
        'ट': 't', 'ठ': 'th', 'ड': 'd', 'ढ': 'dh', 'ण': 'n',
        'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
        'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
        'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v', 'श': 'sh', 'ष': 'sh', 'स': 's', 'ह': 'h',
        'क्ष': 'ksh', 'त्र': 'tr', 'ज्ञ': 'gy',
        '०': '0', '१': '1', '२': '2', '३': '3', '४': '4', '५': '5', '६': '6', '७': '7', '८': '8', '९': '9',
    }
    res = []
    i, n = 0, len(text)
    while i < n:
        c = text[i]
        if c in char_map:
            val = char_map[c]
            is_consonant = c in 'कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहक्षत्रज्ञ'
            if is_consonant:
                if i + 1 < n and text[i+1] == '्':
                    res.append(val)
                    i += 2
                    continue
                elif i + 1 < n and text[i+1] in 'ािीुूृेैोौ':
                    res.append(val + char_map[text[i+1]])
                    i += 2
                    continue
                else:
                    if i + 1 == n:
                        res.append(val)
                    else:
                        res.append(val + 'a')
                    i += 1
            else:
                res.append(val)
                i += 1
        else:
            res.append(c)
            i += 1
    return "".join(res)


def render_text_conditioning(text, font_path, img_size=IMG_SIZE):
    base_size = 80
    try:
        font = ImageFont.truetype(font_path, size=base_size)
    except Exception:
        font = ImageFont.load_default()

    temp = Image.new("L", (1024, 512), color=0)
    draw = ImageDraw.Draw(temp)
    draw.text((50, 50), text, fill=255, font=font)

    np_temp = np.array(temp)
    ys, xs = np.where(np_temp > 0)
    if len(ys) == 0:
        return Image.new("RGB", (img_size, img_size), color="black")

    cropped = temp.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    cw, ch = cropped.size

    max_dim = int(img_size * 0.80)
    scale = min(max_dim / cw, max_dim / ch)
    new_w = max(8, int(cw * scale))
    new_h = max(8, int(ch * scale))
    resized = cropped.resize((new_w, new_h), Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", (img_size, img_size), color="black")
    x = (img_size - new_w) // 2
    y = (img_size - new_h) // 2
    canvas.paste(resized, (x, y))
    return canvas


def make_prompt(text):
    """Dynamic prompt with transliterated word for cross-attention guidance."""
    trans = transliterate_devanagari(text)
    return f"handwritten Devanagari word '{trans}' on white paper, blue ink, high quality"


class DevanagariControlNetDataset(Dataset):
    def __init__(self, hf_ds, tokenizer, font_path):
        self.ds = hf_ds
        self.tokenizer = tokenizer
        self.font_path = font_path
        self.img_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        self.cond_tf = transforms.ToTensor()

        self.valid_indices = []
        for i in range(len(hf_ds)):
            text = str(hf_ds[i].get("text", "")).strip()
            if text:
                self.valid_indices.append(i)
        print(f"  Dataset: {len(self.valid_indices)}/{len(hf_ds)} valid samples")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        i = self.valid_indices[idx]
        try:
            item = self.ds[i]
            text = str(item["text"]).strip() or "नमस्ते"

            real_crop = item["image"].convert("RGB")
            rw, rh = real_crop.size
            aspect = rw / rh

            max_dim = int(IMG_SIZE * 0.80)
            if aspect >= 1:
                tw = min(max_dim, int(rw * (max_dim / max(rw, rh))))
                th = int(tw / aspect)
            else:
                th = min(max_dim, int(rh * (max_dim / max(rw, rh))))
                tw = int(th * aspect)
            tw = max(8, tw)
            th = max(8, th)

            resized_real = real_crop.resize((tw, th), Image.Resampling.LANCZOS)
            target = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(255, 255, 255))
            x = (IMG_SIZE - tw) // 2
            y = (IMG_SIZE - th) // 2
            target.paste(resized_real, (x, y))
            target_tensor = self.img_tf(target)

            cond_img = render_text_conditioning(text, self.font_path)
            cond_tensor = self.cond_tf(cond_img)

            prompt = make_prompt(text)
            input_ids = self.tokenizer(
                prompt, padding="max_length",
                max_length=self.tokenizer.model_max_length,
                truncation=True, return_tensors="pt"
            ).input_ids[0]

            return {
                "pixel_values": target_tensor,
                "conditioning_pixel_values": cond_tensor,
                "input_ids": input_ids,
            }

        except Exception as e:
            cond = self.cond_tf(Image.new("RGB", (IMG_SIZE, IMG_SIZE), "black"))
            tgt = self.img_tf(Image.new("RGB", (IMG_SIZE, IMG_SIZE), (255, 255, 255)))
            ids = self.tokenizer(
                make_prompt("नमस्ते"), padding="max_length",
                max_length=self.tokenizer.model_max_length,
                truncation=True, return_tensors="pt"
            ).input_ids[0]
            return {"pixel_values": tgt, "conditioning_pixel_values": cond, "input_ids": ids}

print("Loading dataset from HuggingFace...")
hf_ds_train = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi", split="train")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5: Model Setup (ControlNet-Only Training)
# ══════════════════════════════════════════════════════════════════

config = ProjectConfiguration(project_dir=OUTPUT_DIR, total_limit=3)
accelerator = Accelerator(
    gradient_accumulation_steps=GRADIENT_ACCUM,
    mixed_precision="fp16",
    project_config=config,
)
set_seed(SEED)

weight_dtype = torch.float16 if accelerator.mixed_precision == "fp16" else torch.float32

print("Loading Stable Diffusion v1.5 components...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")

print("Initializing ControlNet from UNet encoder...")
controlnet = ControlNetModel.from_unet(unet)

# Freeze everything except ControlNet
vae.requires_grad_(False).eval()
text_encoder.requires_grad_(False).eval()
unet.requires_grad_(False).eval()
controlnet.requires_grad_(True).train()

# Gradient checkpointing
controlnet.enable_gradient_checkpointing()

trainable_params = sum(p.numel() for p in controlnet.parameters() if p.requires_grad)
print(f"Trainable params — ControlNet: {trainable_params / 1e6:.1f}M")

optimizer = torch.optim.AdamW(
    controlnet.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-2,
)

train_dataset = DevanagariControlNetDataset(hf_ds_train, tokenizer, FONT_FILE)
train_dataloader = DataLoader(
    train_dataset, shuffle=True, batch_size=BATCH_SIZE,
    num_workers=2, pin_memory=True, persistent_workers=True,
    drop_last=True,
)

lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS * GRADIENT_ACCUM,
    num_training_steps=MAX_TRAIN_STEPS * GRADIENT_ACCUM,
)

controlnet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    controlnet, optimizer, train_dataloader, lr_scheduler
)

text_encoder.to(accelerator.device, dtype=weight_dtype)
vae.to(accelerator.device, dtype=weight_dtype)
unet.to(accelerator.device, dtype=weight_dtype)

cache_path = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path, ignore_errors=True)
    print("🗑️ Cleared HF cache to free disk space.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Model setup complete.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 6: Auto-Resume Logic
# ══════════════════════════════════════════════════════════════════

found_path = None
if os.path.exists("/kaggle/input/"):
    for root, dirs, files in os.walk("/kaggle/input/"):
        for d in dirs:
            if d.startswith("checkpoint-"):
                found_path = os.path.join(root, d)
                break
        if found_path:
            break

if found_path:
    dest_path = os.path.join(OUTPUT_DIR, os.path.basename(found_path))
    if not os.path.exists(dest_path):
        print(f"📦 Copying checkpoint from input: {found_path} → {dest_path}")
        shutil.copytree(found_path, dest_path)

global_step = 0
resume_step = 0
if os.path.exists(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        latest_path = os.path.join(OUTPUT_DIR, latest)
        print(f"📦 Resuming from: {latest_path}")
        try:
            accelerator.load_state(latest_path)
            global_step = int(latest.split("-")[-1])
            resume_step = global_step
            print(f"✅ Resumed from step {global_step}")
        except Exception as e:
            print(f"⚠️ Resume failed: {e}")
            global_step = 0

start_time = time.time()
print(f"🚀 Starting from step: {global_step} | Max steps: {MAX_TRAIN_STEPS}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7: Training Engine
# ══════════════════════════════════════════════════════════════════

loss_history = []
best_loss = float("inf")

print(f"▶️ Training started.")

def generate_samples(step):
    """Generate validation images using in-memory components to prevent dtype/device crash."""
    try:
        unwrapped_cn = accelerator.unwrap_model(controlnet)
        unwrapped_cn.eval()

        # Construct pipeline using existing in-memory weights (prevents CPU/GPU dtype mismatch)
        pipe = StableDiffusionControlNetPipeline(
            vae=vae,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            unet=unet,
            controlnet=unwrapped_cn,
            scheduler=UniPCMultistepScheduler.from_config(noise_scheduler.config),
            safety_checker=None,
            feature_extractor=None,
            requires_safety_checker=False,
        )
        pipe.enable_attention_slicing()

        test_words = ["नमस्ते", "नेपाल", "विकास", "शिक्षा"]
        neg = "blurry, low quality, digital font, typed, outline text"

        for word in test_words:
            cond_img = render_text_conditioning(word, FONT_FILE)
            generator = torch.Generator(device=accelerator.device.type).manual_seed(42)
            prompt = make_prompt(word)
            
            out = pipe(
                prompt=prompt,
                negative_prompt=neg,
                image=cond_img,
                num_inference_steps=20,
                guidance_scale=7.5,
                controlnet_conditioning_scale=1.5,  # Slightly higher for under-trained runs
                generator=generator,
                width=IMG_SIZE,
                height=IMG_SIZE,
            ).images[0]
            out.save(f"{OUTPUT_DIR}/samples/{word}_step{step}.png")
            cond_img.save(f"{OUTPUT_DIR}/samples/{word}_cond.png")

        del pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        unwrapped_cn.train()
        print(f"  📸 Samples saved for step {step}")
    except Exception as e:
        print(f"  ⚠️ Sample generation failed: {e}")
        traceback.print_exc()

controlnet.train()
training_complete = False

while not training_complete:
    for batch in train_dataloader:
        elapsed = time.time() - start_time
        if elapsed > MAX_TIME or global_step >= MAX_TRAIN_STEPS:
            training_complete = True
            break

        with accelerator.accumulate(controlnet):
            try:
                latents = vae.encode(
                    batch["pixel_values"].to(dtype=weight_dtype)
                ).latent_dist.sample() * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (bsz,), device=latents.device,
                ).long()

                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                encoder_hidden_states = text_encoder(
                    batch["input_ids"].to(accelerator.device)
                )[0]

                controlnet_image = batch["conditioning_pixel_values"].to(dtype=weight_dtype)

                down_block_res_samples, mid_block_res_sample = controlnet(
                    noisy_latents, timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                    controlnet_cond=controlnet_image,
                    return_dict=False,
                )

                model_pred = unet(
                    noisy_latents, timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                    down_block_additional_residuals=[
                        s.to(dtype=weight_dtype) for s in down_block_res_samples
                    ],
                    mid_block_additional_residual=mid_block_res_sample.to(dtype=weight_dtype),
                ).sample

                loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad()
                    continue

                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(controlnet.parameters(), 1.0)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()
                    optimizer.zero_grad()
                    continue
                raise

        if accelerator.sync_gradients:
            global_step += 1
            loss_val = loss.detach().item()
            loss_history.append(loss_val)

            if global_step % LOG_STEPS == 0:
                avg = np.mean(loss_history[-100:]) if len(loss_history) >= 100 else np.mean(loss_history)
                lr = optimizer.param_groups[0]["lr"]
                rate = global_step / max(elapsed, 1)
                eta = (MAX_TRAIN_STEPS - global_step) / max(rate, 0.01) / 3600
                print(
                    f"Step {global_step:>6d}/{MAX_TRAIN_STEPS} | "
                    f"Loss: {loss_val:.4f} | Avg100: {avg:.4f} | "
                    f"LR: {lr:.2e} | {rate:.1f} st/s | ETA: {eta:.1f}h"
                )

            if global_step % SAMPLE_STEPS == 0:
                generate_samples(global_step)

            if global_step % SAVE_STEPS == 0:
                save_path = os.path.join(OUTPUT_DIR, f"checkpoint-{global_step}")
                accelerator.save_state(save_path)

                unwrapped_cn = accelerator.unwrap_model(controlnet)
                unwrapped_cn.save_pretrained(os.path.join(save_path, "controlnet"))

                avg_recent = np.mean(loss_history[-200:]) if len(loss_history) >= 200 else np.mean(loss_history)
                if avg_recent < best_loss:
                    best_loss = avg_recent
                    print(f"  🏆 New best avg loss: {best_loss:.4f}")

                all_ckpts = sorted(
                    [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")],
                    key=lambda x: int(x.split("-")[1]),
                )
                for old in all_ckpts[:-2]:
                    shutil.rmtree(os.path.join(OUTPUT_DIR, old), ignore_errors=True)
                print(f"💾 Checkpoint saved: step {global_step}")

if global_step > 0:
    save_path = os.path.join(OUTPUT_DIR, f"checkpoint-{global_step}")
    if not os.path.exists(save_path):
        accelerator.save_state(save_path)
        unwrapped_cn = accelerator.unwrap_model(controlnet)
        unwrapped_cn.save_pretrained(os.path.join(save_path, "controlnet"))

print(f"\n✅ Training complete. Output: {OUTPUT_DIR}/")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8: Post-Training Validation
# ══════════════════════════════════════════════════════════════════
print("Generating final validation samples...")
generate_samples(global_step)
print(f"\n🎉 Done! Check {OUTPUT_DIR}/samples/ for generated images.")